In [32]:
import networkx as nx

# --- 1. 大体3つのコミュニティを狙うLFRグラフの生成 ---
n = 20000

# ※LFRは数学的な制約が厳しいため、エラーが出ないように平均次数なども調整しています
G = nx.LFR_benchmark_graph(
    n=n,
    tau1=2.5,           # 次数分布のべき指数
    tau2=1.5,           # コミュニティサイズのべき指数（小さいほどサイズにばらつきが出る）
    mu=0.1,             # 混合パラメータ（0.1ならコミュニティが明確）
    average_degree=15,   # ノードの平均次数
    min_community=1000,   # 最小のコミュニティサイズ
    max_community=18000,  # 最大のコミュニティサイズ
    seed=10             # シード値（この値を変えると分割のされ方が変わります）
)

# --- 2. 生成された正解データの確認 ---
# 各ノードに割り当てられた 'community' 属性を取得
true_communities = {frozenset(G.nodes[v]['community']) for v in G}

# サイズが大きい順に並び替えて出力
sorted_communities = sorted(true_communities, key=len, reverse=True)

print(f"【LFRが生成した正解】コミュニティ数: {len(sorted_communities)}")
print("-" * 30)
for i, c in enumerate(sorted_communities):
    print(f"  コミュニティ {i+1} のサイズ: {len(c)}")

【LFRが生成した正解】コミュニティ数: 6
------------------------------
  コミュニティ 1 のサイズ: 9114
  コミュニティ 2 のサイズ: 3559
  コミュニティ 3 のサイズ: 3442
  コミュニティ 4 のサイズ: 1591
  コミュニティ 5 のサイズ: 1216
  コミュニティ 6 のサイズ: 1078


In [33]:
import networkx as nx
import numpy as np
import pandas as pd
import os

# --- 1. 大体3つのコミュニティを狙うLFRグラフの生成 ---
n = 20000

G = nx.LFR_benchmark_graph(
    n=n,
    tau1=2.5,           # 次数分布のべき指数
    tau2=1.5,           # コミュニティサイズのべき指数
    mu=0.1,             # 混合パラメータ
    average_degree=15,   # ノードの平均次数
    min_community=1000,   # 最小のコミュニティサイズ
    max_community=18000,  # 最大のコミュニティサイズ
    seed=10             # シード値
)

# --- 2. 生成された正解データの確認 ---
true_communities = {frozenset(G.nodes[v]['community']) for v in G}
sorted_communities = sorted(true_communities, key=len, reverse=True)

print(f"【LFRが生成した正解】コミュニティ数: {len(sorted_communities)}")
print("-" * 30)
for i, c in enumerate(sorted_communities):
    print(f"  コミュニティ {i+1} のサイズ: {len(c)}")


# --- 3. コミュニティごとにJSON隣接行列形式でファイル保存 ---

# 新しく隣接行列JSONを保存する場所 (ご提示のコードに合わせる)
OUT_FOLDER = "LFR_communities"

# 1. 出力フォルダを作成
os.makedirs(OUT_FOLDER, exist_ok=True)
print(f"\n✅ 出力フォルダを作成しました: {OUT_FOLDER}")

processed_count = 0

for i, c in enumerate(sorted_communities):
    # ファイル名を変更し、拡張子を .txt に統一
    output_filename = f"LFR_community_{i+1}.txt"
    output_path = os.path.join(OUT_FOLDER, output_filename)
    
    # コミュニティ内のノードだけを抽出して部分グラフを作成
    subgraph = G.subgraph(c)
    
    if subgraph.number_of_nodes() == 0:
        print(f"警告: コミュニティ {i+1} はノード数が0のためスキップしました。")
        continue

    try:
        # 2. NetworkXグラフを隣接行列（NumPy配列）に変換
        # ※LFRは無向グラフですが、同様に隣接行列化可能です
        adj = nx.to_numpy_array(subgraph) 
        
        # 3. データ型をint32に変換
        adj = adj.astype(np.int32)
        
        # 4. Pandas DataFrameに変換
        # ノードIDは含まれず、行/列のインデックスが0, 1, 2...となる行列形式
        df = pd.DataFrame(adj)
        
        # 5. JSON形式でファイルに保存
        # デフォルト(orient='columns')で保存され、インデックスがキーになります。
        df.to_json(output_path) 
        
        print(f"  - 💾 保存成功: {output_filename} ({subgraph.number_of_nodes()}ノード)")
        processed_count += 1

    except Exception as e:
        print(f"  - ❌ 保存失敗: コミュニティ {i+1} - エラー: {e}")

print(f"\n--- 完了 ---")
print(f"合計 {len(sorted_communities)} 件中、{processed_count} 件のコミュニティをJSON隣接行列形式で保存しました。")

【LFRが生成した正解】コミュニティ数: 6
------------------------------
  コミュニティ 1 のサイズ: 9114
  コミュニティ 2 のサイズ: 3559
  コミュニティ 3 のサイズ: 3442
  コミュニティ 4 のサイズ: 1591
  コミュニティ 5 のサイズ: 1216
  コミュニティ 6 のサイズ: 1078

✅ 出力フォルダを作成しました: LFR_communities
  - 💾 保存成功: LFR_community_1.txt (9114ノード)
  - 💾 保存成功: LFR_community_2.txt (3559ノード)
  - 💾 保存成功: LFR_community_3.txt (3442ノード)
  - 💾 保存成功: LFR_community_4.txt (1591ノード)
  - 💾 保存成功: LFR_community_5.txt (1216ノード)
  - 💾 保存成功: LFR_community_6.txt (1078ノード)

--- 完了 ---
合計 6 件中、6 件のコミュニティをJSON隣接行列形式で保存しました。


In [12]:
import networkx as nx

n = 10000
target_communities = 3
ideal_G = None
ideal_seed = None

print("理想的な3つのコミュニティを持つグラフを探索しています...")

# シード値を1から順番に試す（最大1000回）
for current_seed in range(1, 1000):
    try:
        # LFRグラフの生成
        G = nx.LFR_benchmark_graph(
            n=n, tau1=3.0, tau2=1.5, mu=0.1, average_degree=15,
            min_community=500, max_community=8000, seed=current_seed
        )
        
        # コミュニティ数の確認
        true_communities = {frozenset(G.nodes[v]['community']) for v in G}
        
        # ピッタリ3つに分かれたらループを終了
        if len(true_communities) == target_communities:
            ideal_G = G
            ideal_seed = current_seed
            print(f"成功！ Seed: {current_seed} で {target_communities} つのコミュニティが生成されました。")
            
            # コミュニティサイズの出力
            sizes = sorted([len(c) for c in true_communities], reverse=True)
            print(f"各コミュニティのサイズ: {sizes}")
            break
            
    except nx.NetworkXError:
        # LFRはパラメータと乱数の組み合わせにより数学的に生成不可能な場合があり、
        # その際に出るエラーを無視して次のシード値を試します。
        continue

# --- 保存処理 ---
if ideal_G is not None:
    # 【重要】ファイル保存のための前処理
    # set型は保存できないため、扱いやすい整数（int）のIDに変換します
    for node in ideal_G.nodes():
        # 所属するコミュニティのIDを取り出す
        comm_set = ideal_G.nodes[node]['community']
        ideal_G.nodes[node]['community_id'] = list(comm_set)[0]
        # 保存エラーの原因になる元のset型属性を削除
        del ideal_G.nodes[node]['community'] 
        
    # GMLフォーマットで保存（Gephiなどの外部ツールでも読み込める標準形式）
    filepath = "lfr_3_communities.gml"
    nx.write_gml(ideal_G, filepath)
    print(f"\nネットワークを保存しました: {filepath}")
    
else:
    print("指定した回数内で、3つのコミュニティになるシード値が見つかりませんでした。")

理想的な3つのコミュニティを持つグラフを探索しています...


ExceededMaxIterations: Could not create power law sequence

In [11]:
import networkx as nx

n = 20000

try:
    print("緩和されたLFRネットワークを生成しています...")
    G = nx.LFR_benchmark_graph(
        n=n,
        tau1=3.0,           # 変更：2.5 -> 3.0 (次数の極端なばらつきを少し抑え、計算を安定させる)
        tau2=1.5,
        mu=0.1,
        average_degree=10,  # 変更：15 -> 10 (エッジ数を減らして生成しやすくする)
        max_degree=1000,    # 追加：最大次数を明示的に制限することでエラーを回避
        min_community=1000, # 変更：4000 -> 1000 (小さいコミュニティの存在を許容する)
        max_community=10000,# 変更：12000 -> 10000
        seed=42,
        max_iters=5000      # 追加：内部計算の諦め時間を少し長くする
    )
    print("生成成功！")
except nx.NetworkXError as e:
    print(f"エラー発生: {e}")

緩和されたLFRネットワークを生成しています...
生成成功！
